# Demo how to use cryptography for secure caching
Demonstrates
- EncryptedCache

In [ ]:
%load_ext autoreload
%autoreload 2
from __future__ import annotations

from cryptography.fernet import Fernet
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
import base64
import hashlib
import logging

from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Callable, Union
import json
import os
import re

logger = logging.getLogger("akepylib.encrypted_cache")

# Types for type hints
JsonData = Any


def get_hashed_filename(text: str, length: int = 32, prefix: str | None = None) -> str:
    """Return a hex digest suitable as a filename."""
    if length < 8 or length > 64:
        raise ValueError("length must be between 8 and 64")
    digest = hashlib.sha256(text.encode("utf-8")).hexdigest()
    trimmed = digest[:length]
    if prefix is None:
        return trimmed
    safe_prefix = re.sub(r"[^a-z0-9_-]+", "-", prefix.lower()).strip("-")
    if not safe_prefix:
        raise ValueError("prefix must contain at least one alphanumeric character")
    return f"{safe_prefix}-{trimmed}"


class SaltMismatchError(ValueError):
    """Raised when the salt used to encrypt does not match the current salt."""


class EncryptedCache:
    """Encrypts and stores payloads on disk with cleartext metadata."""

    DEFAULT_CACHE_BASE_DIRECTORY = Path("data") / "encryptedcache"
    DEFAULT_TTL = "5 days"


    def _project_root(self) -> Path | None:
        """Find project root by locating pyproject.toml."""
        current = Path.cwd().resolve()
        for parent in [current, *current.parents]:
            if (parent / "pyproject.toml").is_file():
                return parent
        return None


    def _default_cache_base_directory(self) -> Path:
        """Pick a cache directory based on project-root availability."""
        project_root = self._project_root()
        if project_root is not None:
            return project_root / self.DEFAULT_CACHE_BASE_DIRECTORY
        else:
            fallback = Path(f"/tmp/encryptedcache-{os.getuid()}")
            logger.warning("No pyproject.toml found, using fallback cache dir: %s", fallback)
            return fallback


    @staticmethod
    def _salt_fingerprint(salt: bytes) -> str:
        """Return a short hex fingerprint of the salt for envelope storage."""
        return hashlib.sha256(salt).hexdigest()[:16]


    def __init__(
            self,
            secrets: dict[str, str],
            salt: bytes,
            suffix: str = ".enc.json",
            cache_base_directory: Union[str, Path] | None = None,
            ttl: datetime | str | None = None,
        ):
            """Initialize cache ciphers and defaults."""
            self._salt = salt
            self._salt_fp = self._salt_fingerprint(salt)
            self.ciphers = {}
            self._suffix = suffix
            base_dir = Path(cache_base_directory) if cache_base_directory else self._default_cache_base_directory()
            self.cache_base_directory = base_dir
            self.default_ttl = ttl if ttl is not None else self.DEFAULT_TTL
            for name, password in secrets.items():
                logger.debug("Deriving key for %r", name)
                key = self._derive_key(password, salt)
                self.ciphers[name] = Fernet(key)
            logger.info(
                "Initialized EncryptedCache with %d key(s), base_dir=%s, default_ttl=%s",
                len(self.ciphers), self.cache_base_directory, self.default_ttl,
            )


    def _ensure_secure_dir(self, path: Path) -> None:
        """Ensure directory exists with user-only permissions."""
        path.mkdir(parents=True, exist_ok=True)
        os.chmod(path, 0o700)


    def _write_secure_json(self, filepath: Path, payload: dict[str, Any]) -> None:
        """Write JSON file with user-only permissions."""
        self._ensure_secure_dir(filepath.parent)
        fd = os.open(filepath, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
        with os.fdopen(fd, "w", encoding="utf-8") as handle:
            json.dump(payload, handle)


    def _derive_key(self, password: str, salt: bytes) -> bytes:
        """Derive a Fernet key from the given password and salt."""
        kdf = PBKDF2HMAC(
            algorithm=hashes.SHA256(),
            length=32,
            salt=salt,
            iterations=480000,  # OWASP recommendation 2023
        )
        return base64.urlsafe_b64encode(kdf.derive(password.encode()))


    def _normalize_path(self, filepath: Union[str, Path]) -> Path:
        """Normalize the file path to use the configured suffix."""
        path = Path(filepath)
        name = path.name
        if name.endswith(self._suffix):
            return path
        for suffix in (self._suffix, ".enc.json", ".enc.xml", ".enc"):
            if name.endswith(suffix):
                name = name[: -len(suffix)]
                result = path.with_name(name + self._suffix)
                logger.debug("Normalized path: %s -> %s", filepath, result)
                return result
        result = path.with_name(name + self._suffix)
        logger.debug("Normalized path: %s -> %s", filepath, result)
        return result


    def _resolve_cache_path(self, cache_id: str, cache_base_directory: Union[str, Path] | None) -> Path:
        """Resolve cache path from the base directory and relative cache id."""
        base_dir = Path(cache_base_directory) if cache_base_directory else self.cache_base_directory
        cache_path = Path(cache_id)
        if cache_path.is_absolute():
            raise ValueError("cache_id must be relative")
        if not cache_path.parts or cache_path.parts == (".",):
            raise ValueError("cache_id must not be empty")
        if ".." in cache_path.parts:
            raise ValueError("cache_id must not contain path traversal")
        # stricter check whether the resolved path is below the base_dir
        candidate = (base_dir / cache_path).resolve(strict=False)
        base_resolved = base_dir.resolve(strict=False)
        if not candidate.is_relative_to(base_resolved):
            raise ValueError("cache_id must stay within cache_base_directory")
        resolved = self._normalize_path(candidate)
        logger.debug("Resolved cache path: %s", resolved)
        return resolved


    def _format_datetime(self, value: datetime, label: str) -> str:
        """Format a timezone-aware datetime as ISO-8601."""
        if value.tzinfo is None or value.tzinfo.utcoffset(value) is None:
            raise ValueError(f"{label} must be timezone-aware")
        return value.isoformat()


    def _parse_datetime(self, value: str, label: str) -> datetime:
        """Parse an ISO-8601 datetime string, assuming UTC for naive values."""
        try:
            parsed = datetime.fromisoformat(value)
        except ValueError as exc:
            raise ValueError(f"{label} must be ISO-8601, got {value!r}") from exc
        if parsed.tzinfo is None or parsed.tzinfo.utcoffset(parsed) is None:
            return parsed.replace(tzinfo=timezone.utc)
        return parsed


    def _parse_duration(self, value: str) -> timedelta | None:
        """Parse a human-readable duration string into a timedelta, e.g.
        "3d 5h 30m 30s" or "3d" or "infinity"
        """
        normalized = value.strip().lower()
        if normalized in {"infinite", "inf", "forever"}:
            return None
        tokens = re.findall(r"(\d+)\s*(days?|d|hours?|h|minutes?|mins?|m|seconds?|secs?|s)", normalized)
        if not tokens:
            raise ValueError(f"Invalid duration string: {value!r}")

        total = timedelta()
        for amount_str, unit in tokens:
            amount = int(amount_str)
            if unit in {"d", "day", "days"}:
                total += timedelta(days=amount)
            elif unit in {"h", "hour", "hours"}:
                total += timedelta(hours=amount)
            elif unit in {"m", "min", "mins", "minute", "minutes"}:
                total += timedelta(minutes=amount)
            elif unit in {"s", "sec", "secs", "second", "seconds"}:
                total += timedelta(seconds=amount)
        logger.debug("Parsed duration %r -> %s", value, total)
        return total


    def _is_cache_valid(self, created_at: str | None, ttl: datetime | str | int | None) -> bool:
        """Return whether the cache entry should be used based on TTL.

        ttl=None means infinity (always valid), ttl=0 means always stale.
        """
        if ttl is None:
            valid = True
        elif isinstance(ttl, int) and ttl == 0:
            valid = False
        elif isinstance(ttl, datetime):
            now = datetime.now(timezone.utc)
            expires_at = ttl
            if expires_at.tzinfo is None or expires_at.utcoffset() is None:
                expires_at = expires_at.replace(tzinfo=timezone.utc)
            valid = now <= expires_at
        elif isinstance(ttl, str):
            now = datetime.now(timezone.utc)
            try:
                expires_at = self._parse_datetime(ttl, "ttl")
                valid = now <= expires_at
            except ValueError:
                duration = self._parse_duration(ttl)
                if duration is None:
                    valid = True
                elif created_at is None:
                    valid = False
                else:
                    created_dt = self._parse_datetime(created_at, "created_at")
                    valid = now <= created_dt + duration
        else:
            raise TypeError("ttl must be datetime, str, int(0), or None")
        logger.debug("Cache validity check: created_at=%s, ttl=%s, valid=%s", created_at, ttl, valid)
        return valid


    def _check_salt_fingerprint(self, payload: dict[str, Any], filepath: Path) -> None:
        """Raise SaltMismatchError if the envelope salt doesn't match ours."""
        stored_fp = payload.get("salt_sha256")
        logger.debug("Salt fingerprint check: stored=%s, current=%s", stored_fp, self._salt_fp)
        if stored_fp is not None and stored_fp != self._salt_fp:
            raise SaltMismatchError(
                f"Salt mismatch for {filepath.name}: "
                f"file has {stored_fp}, current is {self._salt_fp}"
            )


    def save(
        self,
        filepath: Union[str, Path],
        data: bytes,
        key_name: str = "default",
        validasof_datetime: datetime | None = None,
        comment: str | None = None,
    ) -> Path:
        """Encrypt data and write it with metadata to disk."""
        if key_name not in self.ciphers:
            raise ValueError(f"Unknown key: {key_name}")

        filepath = self._normalize_path(filepath)
        inner_payload = {
            "data_b64": base64.b64encode(data).decode("ascii"),
        }
        if validasof_datetime is not None:
            inner_payload["validasof_datetime"] = self._format_datetime(validasof_datetime, "validasof_datetime")

        encrypted = self.ciphers[key_name].encrypt(json.dumps(inner_payload).encode("utf-8"))

        payload = {
            "key": key_name,
            "salt_sha256": self._salt_fp,
            "created_at": self._format_datetime(datetime.now(timezone.utc), "created_at"),
            "encrypted": encrypted.decode("utf-8"),
        }
        if comment is not None:
            payload["comment"] = comment

        self._write_secure_json(filepath, payload)
        logger.info("Saved encrypted cache: %s (key=%s)", filepath, key_name)
        return filepath


    def load(self, filepath: Union[str, Path]) -> bytes:
        """Load and decrypt data from the cache file."""
        data, _meta = self.load_entry(filepath)
        return data


    def load_entry(self, filepath: Union[str, Path]) -> tuple[bytes, dict[str, str | None]]:
        """Load data plus metadata from the cache file.

        Raises SaltMismatchError if the file was written with a different salt.
        """
        filepath = self._normalize_path(filepath)
        payload = json.loads(filepath.read_text())
        key_name = payload["key"]

        self._check_salt_fingerprint(payload, filepath)

        if key_name not in self.ciphers:
            raise ValueError(f"Key '{key_name}' not available")

        encrypted = payload["encrypted"].encode("utf-8")
        inner_payload = json.loads(self.ciphers[key_name].decrypt(encrypted).decode("utf-8"))
        data = base64.b64decode(inner_payload["data_b64"])
        validasof_datetime = inner_payload.get("validasof_datetime")

        metadata = {
            "key": key_name,
            "created_at": payload.get("created_at"),
            "comment": payload.get("comment"),
            "validasof_datetime": validasof_datetime,
        }
        logger.info("Loaded cache entry: %s (key=%s)", filepath, key_name)
        return data, metadata


    def exists(self, filepath: Union[str, Path]) -> bool:
        """Check whether the cache file exists."""
        filepath = self._normalize_path(filepath)
        return filepath.exists()


    def execute_cached(
        self,
        key_name: str,
        callback: Callable[[], JsonData],
        cache_id: str,
        rerun: bool = False,
        ttl: datetime | str | int | None = None,
        cache_base_directory: Union[str, Path] | None = None,
        validasof_datetime: datetime | None = None,
        comment: str | None = None,
    ) -> JsonData:
        """Load data from cache or run callback and store the result.

        On SaltMismatchError, the cached file is treated as stale and the
        callback is re-executed.
        """
        cache_path = self._resolve_cache_path(cache_id, cache_base_directory)
        effective_ttl = ttl if ttl is not None else self.default_ttl

        if rerun:
            logger.warning("Forced rerun for %s, ignoring cached data", cache_path)
        elif cache_path.exists():
            try:
                data, metadata = self.load_entry(cache_path)
            except SaltMismatchError:
                logger.warning("Salt mismatch for %s, treating as cache miss", cache_path)
            else:
                if self._is_cache_valid(metadata.get("created_at"), effective_ttl):
                    logger.info("Cache hit for %s", cache_path)
                    return json.loads(data)
                else:
                    logger.info("Cache expired for %s, re-executing callback", cache_path)

        if not rerun:
            logger.info("Cache miss for %s, executing callback", cache_path)

        result = callback()
        self._validate_json_data(result)
        payload = json.dumps(result).encode("utf-8")
        self.save(cache_path, payload, key_name, validasof_datetime=validasof_datetime, comment=comment)
        return result


    def _validate_json_data(self, value: JsonData) -> None:
        """Ensure the cached value is JSON-serializable."""
        try:
            json.dumps(value)
        except TypeError as exc:
            raise ValueError("callback must return JSON-serializable data") from exc

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import logging

# Configure logging for the demo — set to DEBUG to see all messages, INFO for operational only
logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
logging.getLogger("akepylib.encrypted_cache").setLevel(logging.INFO)


secrets = {
    "MyTestSecret": "MyTestSecretKey",
    "AnotherSecret": "AnotherSecretKey",
}

cache = EncryptedCache(secrets, salt=b"demo-crypto")

# Cache miss handling
cache_id = "api_response"

def fetch_data():
    return {"value": "some data that we want to cache"}

result = cache.execute_cached(
    "MyTestSecret",
    fetch_data,
    cache_id,
    ttl="5 days",
    validasof_datetime=datetime.now(timezone.utc),
    comment="daily refresh",
)
print(result)

In [ ]:
def callback_callee(d: datetime, name: str) -> str:
    return f"Hello {name} at {d.strftime("%Y-%m-%d")} - I will now retrieve your code"


def get_callback(aname, dnow):
    def innercallback():
        return callback_callee(dnow, aname)
    return innercallback


aname = "Ake"
dnow = datetime.now(timezone.utc)
callback = get_callback(aname, dnow)

callback_description = f"get_callback(aname={aname}, dnow={dnow.strftime("%Y-%m-%d")})"
hashed_id = get_hashed_filename(callback_description, prefix="cd-demo")

result_hashed = cache.execute_cached(
    "MyTestSecret",
    callback,
    hashed_id,
    ttl="5 days",
    validasof_datetime=dnow,
    comment="hashed filename demo",
)
print(result_hashed)